In [42]:
# generate_data
import pandas as pd

def generate_synthetic_datasets() -> None:
    # 1. Dataset de Pacientes
    pacientes_data = {
        "patient_id": ["45678912", "78912345", "12345678", "98765432"],
        "name": ["Carlos Mendoza", "Ana López", "Juan Pérez", "Sofía Rodríguez"],
        "glucose_level": [45, 280, 110, 65],  # Emergencia, Urgencia, Agendar Cita, Emergencia
        "last_visit_date": ["2026-03-15", "2026-04-02", "2026-05-10", "2026-05-16"]
    }
    pd.DataFrame(pacientes_data).to_csv("content/dataset04.csv", index=False)
    print("Dataset 'dataset04.csv' generado exitosamente.")

    # 2. Dataset de Médicos Disponibles (Formato CSV solicitado)
    medicos_data = {
        "medico_id": ["M001", "M002", "M003", "M004"],
        "nombre_medico": ["Dra. Elena Rostova", "Dr. Alan Grant", "Dra. Sarah Connor", "Dr. Henry Wu"],
        "especialidad": ["Endocrinología Pediátrica", "Diabetología Adultos", "Endocrinología General", "Trastornos Metabólicos"],
        "dias_atencion": ["Lunes a Viernes", "Lunes a Viernes", "Sábado", "Domingo"]
    }
    pd.DataFrame(medicos_data).to_csv("content/medicos04.csv", index=False)
    print("Dataset 'medicos04.csv' generado exitosamente.")

if __name__ == "__main__":
    generate_synthetic_datasets()

Dataset 'dataset04.csv' generado exitosamente.
Dataset 'medicos04.csv' generado exitosamente.


In [49]:
# 1. Encargado de inicializar las variables de entorno de forma segura.
import os
from dotenv import load_dotenv
from langsmith import traceable

if "SSL_CERT_FILE" in os.environ:
    del os.environ["SSL_CERT_FILE"]
    
def load_configurations() -> None:
    """
    Carga las variables de entorno desde el archivo .env e inicializa 
    las configuraciones globales del sistema.
    """
    load_dotenv()
    
    # Verificación preventiva de seguridad para entornos críticos
    if not os.getenv("OPENAI_API_KEY"):
        raise ValueError(
            "CRÍTICO: La variable de entorno 'OPENAI_API_KEY' no está configurada.\n"
            "Por favor, asegúrese de crear un archivo '.env' en la raíz con dicha credencial."
        )

In [50]:
# agent_logic
from typing import List
import pandas as pd
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage

# ==========================================
# REGLAS Y PAYLOADS DE AGENDA (CÓDIGO DE BASE)
# ==========================================
HORARIOS_REGLAS = """
REGLAS DE HORARIOS DISPONIBLES:
- Lunes a Viernes: 07:00, 09:00, 11:00, 13:00, 15:00, 17:00, 19:00
- Sábado: 07:00, 09:00, 11:00, 13:00, 15:00, 17:00, 19:00
- Domingo: 09:00, 11:00, 13:00, 15:00, 17:00
"""

@tool
def notify_doctor_emergency(patient_info: str) -> str:
    """Notifica inmediatamente al médico de cabecera sobre el estado crítico de emergencia del paciente."""
    return "EMERGENCIA: Los datos proporcionados están siendo notificados a tu médico, se recomienda llamar a emergencias."

@tool
def confirm_appointment(appointment_details: str) -> str:
    """Registra y confirma formalmente la cita de seguimiento en la agenda de la clínica."""
    return f"CITA CONFIRMADA: Su espacio ha sido reservado exitosamente. Detalles: {appointment_details}"

In [51]:
# ==========================================
# AGENTE 1: FLUJO DE EMERGENCIA
# ==========================================
class EmergencyAgent:
    def __init__(self) -> None:
        self.llm = ChatOpenAI(model="gpt-4o", temperature=0.0)
        self.tools = [notify_doctor_emergency]
        self.agent_chain = ChatPromptTemplate.from_messages([
            ("system", (
                "Eres un asistente de triaje de Endocrinología.\n"
                "El paciente está cruzando un cuadro de HIPOGLUCEMIA GRAVE. Tu tono debe ser calmado, empático pero sumamente directivo.\n\n"
                "PROTOCOLO DE RECOPILACIÓN OBLIGATORIO:\n"
                "Pregunta interactivamente:\n"
                "1. ¿Cuáles son sus síntomas actuales exactos?\n"
                "2. ¿Cuántas veces a la semana hizo ejercicio?\n"
                "3. ¿Consumió carbohidratos en exceso o bebió alcohol recientemente?\n\n"
                "Instrucción: Brinda pautas breves de mitigación (comer 15g carbohidratos rápidos). Al responder los 3 puntos, ejecuta obligatoriamente `notify_doctor_emergency`."
            )),
            MessagesPlaceholder(variable_name="chat_history"),
        ]) | self.llm.bind_tools(self.tools)

    def start_conversation_loop(self, patient_name: str, glucose: float) -> None:
        print("\n" + "="*60)
        print(f" ALERTA CRÍTICA: INICIANDO CHAT DE EMERGENCIA - {patient_name}")
        print("="*60)
        chat_history: List[BaseMessage] = []
        initial_input = f"Hola {patient_name}, detectamos un nivel de glucosa crítico ({glucose} mg/dL). Por favor, indícame qué síntomas tienes ahora mismo."
        print(f"\n[Agente]: {initial_input}")
        chat_history.append(AIMessage(content=initial_input))
        
        while True:
            user_input = input("\n[Paciente]: ").strip()
            if not user_input: continue
            chat_history.append(HumanMessage(content=user_input))
            response = self.agent_chain.invoke({"chat_history": chat_history})
            
            if response.tool_calls:
                for tool_call in response.tool_calls:
                    if tool_call["name"] == "notify_doctor_emergency":
                        print(f"\n>>> {notify_doctor_emergency.invoke(tool_call['args'])} <<< \n")
                        return
            print(f"\n[Agente]: {response.content}")
            chat_history.append(AIMessage(content=response.content))

In [52]:
# ==========================================
# AGENTE 2: FLUJO DE AGENDAMIENTO
# ==========================================
class AppointmentAgent:
    def __init__(self) -> None:
        self.llm = ChatOpenAI(model="gpt-4o", temperature=0.0)
        self.tools = [confirm_appointment]
        
        # Carga dinámica del catálogo de médicos en formato CSV
        df_medicos = pd.read_csv("content/medicos04.csv")
        medicos_context = df_medicos.to_markdown(index=False)

        self.agent_chain = ChatPromptTemplate.from_messages([
            ("system", (
                "Eres el asistente virtual encargado de agendar citas de seguimiento preventivo en Diabetes.\n"
                "Tu objetivo es guiar de forma amable al paciente para que elija un médico y un horario válido.\n\n"
                f"{HORARIOS_REGLAS}\n"
                f"CATÁLOGO DE MÉDICOS DISPONIBLES (CSV extraído):\n{medicos_context}\n\n"
                "REGLAS DE AGENDAMIENTO:\n"
                "- Muestra de forma clara la lista de médicos con sus especialidades y los rangos de días de atención.\n"
                "- Oferta los horarios según los intervalos de 2 horas según el día que el paciente prefiera (L-V, S o D).\n"
                "- El paciente debe elegir un médico específico y un horario válido que coincida con los días de atención de dicho médico.\n"
                "- Una vez el paciente defina (Médico, Día y Hora), ejecuta inmediatamente la herramienta `confirm_appointment` para consolidar la reserva."
            )),
            MessagesPlaceholder(variable_name="chat_history"),
        ]) | self.llm.bind_tools(self.tools)

    def start_conversation_loop(self, patient_name: str) -> None:
        print("\n" + "="*60)
        print(f" INTERFAZ DE AGENDAMIENTO DE CITAS: {patient_name}")
        print("="*60)
        chat_history: List[BaseMessage] = []
        initial_input = f"¡Hola {patient_name}! Tu nivel de glucosa está en metas de control diario. Vamos a agendar tu cita de seguimiento. Aquí tienes nuestros médicos y horarios. ¿Qué día te convendría asistir (Lunes a Viernes, Sábado o Domingo)?"
        print(f"\n[Agente]: {initial_input}")
        chat_history.append(AIMessage(content=initial_input))
        
        while True:
            user_input = input("\n[Paciente]: ").strip()
            if not user_input: continue
            chat_history.append(HumanMessage(content=user_input))
            response = self.agent_chain.invoke({"chat_history": chat_history})
            
            if response.tool_calls:
                for tool_call in response.tool_calls:
                    if tool_call["name"] == "confirm_appointment":
                        print(f"\n>>> {confirm_appointment.invoke(tool_call['args'])} <<< \n")
                        return
            print(f"\n[Agente]: {response.content}")
            chat_history.append(AIMessage(content=response.content))

In [53]:
# triager
import pandas as pd
from typing import Tuple
#from config import load_configurations  # <-- Importación de la configuración
#from agent_logic import EmergencyAgent, AppointmentAgent

# Pauta de reglas de laboratorio en formato Markdown
REGLAS_ORDEN_LABORATORIO_MD = """
### 📋 ORDEN DE LABORATORIO: REQUISITOS DE PREPARACIÓN
Por favor, siga estrictamente las siguientes pautas antes de acudir a la toma de muestras:

1. **Glucosa Basal:**
   * ⚠️ **Requisito Obligatorio:** Requiere un ayuno estricto de 8 a 10 horas. No consuma alimentos ni bebidas (salvo agua pura) durante este lapso.
2. **Hemoglobina Glicosilada (HbA1c):**
   * ✅ **Requisito:** No necesita ayuno. Esta prueba refleja tu promedio de azúcar en la sangre durante los últimos 3 meses.
3. **Examen de Orina y Perfil Lipídico:**
   * ⚠️ **Requisito Obligatorio:** Generalmente requieren de 8 a 12 horas de ayuno previo para no alterar los parámetros de colesterol y triglicéridos.
"""

class DeterministicTriager:
    def __init__(self, csv_path: str = "content/dataset04.csv") -> None:
        self.csv_path = csv_path
        self._load_dataset()

    def _load_dataset(self) -> None:
        """Carga el archivo CSV de pacientes en un DataFrame indexado."""
        try:
            self.df = pd.read_csv(self.csv_path)
            self.df["patient_id"] = self.df["patient_id"].astype(str)
        except FileNotFoundError:
            raise FileNotFoundError(f"Error: No se encontró el archivo '{self.csv_path}'. Ejecute 'generate_data.py'.")

    def evaluate_triage(self, glucose: float) -> Tuple[str, str]:
        """
        Aplica las reglas deterministas basadas en la matriz de negocio de Markdown.
        Retorna: (Categoría, Mensaje de Acción)
        """
        # Regla 1: < 70 mg/dL (Severe Hypoglycemia) -> EMERGENCIA
        if glucose < 70.0:
            return "EMERGENCIA", "Llamar a Emergencias"
        
        # Regla 2: > 250 mg/dL (Severe Hyperglycemia) -> URGENCIAS
        elif glucose > 250.0:
            return "URGENCIAS", "Solicitar Orden Laboratorio"
        
        # Regla 3: 70 - 250 mg/dL (Daily Control Goal) -> AGENDAR CITA
        else:
            return "AGENDAR CITA", "Felicidades: Agendar tu cita de seguimiento"

    def run_triage(self) -> None:
        """Punto de entrada principal para solicitar la identificación y procesar el triaje."""
        print("\n--- BIENVENIDO AL SISTEMA DE TRIAJE DIGITAL (ENDOCRINOLOGÍA) ---")
        patient_id_input = input("Ingrese el DNI o CÓDIGO PACIENTE: ").strip()
        
        if not patient_id_input:
            print("Error: El código ingresado no puede estar vacío.")
            return

        # Búsqueda determinista del paciente en el DataFrame
        patient_record = self.df[self.df["patient_id"] == patient_id_input]
        if patient_record.empty:
            print(f"Error: El paciente con ID '{patient_id_input}' no está registrado.")
            return

        patient_name = str(patient_record.iloc[0]["name"])
        glucose_level = float(patient_record.iloc[0]["glucose_level"])
        
        print(f"\n[Sistema]: Paciente localizado: {patient_name}")
        print(f"[Sistema]: Último registro de glucosa: {glucose_level} mg/dL")
        
        category, action_message = self.evaluate_triage(glucose_level)
        
        print(f"\n[RESULTADO TRIAJE]: Categoría -> **{category}**")
        print(f"[ACCIÓN DETERMINISTA]: {action_message}")
        
        # Orquestación de Flujo del Sistema según la Categoría Evaluada
        if category == "EMERGENCIA":
            agent_emergencia = EmergencyAgent()
            agent_emergencia.start_conversation_loop(patient_name, glucose_level)
            
        elif category == "URGENCIAS":
            print(REGLAS_ORDEN_LABORATORIO_MD)
            print("[Sistema]: Flujo de Urgencias finalizado. Sesión Cerrada.\n")
            
        elif category == "AGENDAR CITA":
            agent_agenda = AppointmentAgent()
            agent_agenda.start_conversation_loop(patient_name)

In [55]:
if __name__ == "__main__":
    try:
        # 1. Inicialización determinista y carga de variables de entorno globales (.env)
        load_configurations()
        
        # 2. Instanciación y arranque del motor de triaje
        triager = DeterministicTriager()
        triager.run_triage()
    except Exception as e:
        print(f"\nError durante la ejecución del sistema: {e}")


--- BIENVENIDO AL SISTEMA DE TRIAJE DIGITAL (ENDOCRINOLOGÍA) ---

[Sistema]: Paciente localizado: Juan Pérez
[Sistema]: Último registro de glucosa: 110.0 mg/dL

[RESULTADO TRIAJE]: Categoría -> **AGENDAR CITA**
[ACCIÓN DETERMINISTA]: Felicidades: Agendar tu cita de seguimiento

 INTERFAZ DE AGENDAMIENTO DE CITAS: Juan Pérez

[Agente]: ¡Hola Juan Pérez! Tu nivel de glucosa está en metas de control diario. Vamos a agendar tu cita de seguimiento. Aquí tienes nuestros médicos y horarios. ¿Qué día te convendría asistir (Lunes a Viernes, Sábado o Domingo)?

[Agente]: ¡Perfecto! Aquí tienes la lista de médicos disponibles para el viernes:

1. **Dra. Elena Rostova** - Especialidad: Endocrinología Pediátrica
2. **Dr. Alan Grant** - Especialidad: Diabetología Adultos

Los horarios disponibles para el viernes son: 07:00, 09:00, 11:00, 13:00, 15:00, 17:00, 19:00.

Por favor, elige un médico y un horario que te convenga.

>>> CITA CONFIRMADA: Su espacio ha sido reservado exitosamente. Detalles: Dr

---

### Casos de Prueba Listos para Validar el Rigor Endocrinologo:

* **Caso URGENCIAS:** Ingresa el DNI `78912345` (Glucosa: 280 mg/dL). 
* **Caso AGENDAR CITA:** Ingresa el DNI `12345678` (Glucosa: 110 mg/dL).
* **Caso EMERGENCIA (Híbrido Agéntico):** Ingresa el DNI `45678912` (Glucosa: 45 mg/dL). 

A continuación, se presentan las simulaciones exactas de la salida en consola para los tres escenarios clínicos del **Asistente Virtual Médico IA (Triaje Endocrino)**, considerando la integración de las nuevas reglas de negocio de laboratorio (para Urgencias) y el flujo interactivo de agenda con el catálogo de médicos en CSV (para Agendar Cita).

---

### Caso 1: URGENCIAS (Glucosa: 280 mg/dL)

* **Datos de Entrada:** Paciente con DNI `78912345` (Ana López).
* **Comportamiento Esperado:** El sistema detecta hiperglicemia severa de forma determinista, imprime el mensaje de acción obligatorio y renderiza de inmediato las instrucciones específicas de preparación en Markdown. El script termina de forma segura sin consumir tokens de IA.

```text
--- BIENVENIDO AL SISTEMA DE TRIAJE DIGITAL (ENDOCRINOLOGÍA) ---

[Sistema]: Paciente localizado: Ana López
[Sistema]: Último registro de glucosa: 280.0 mg/dL

[RESULTADO TRIAJE]: Categoría -> **URGENCIAS**
[ACCIÓN DETERMINISTA]: Solicitar Orden Laboratorio

### 📋 ORDEN DE LABORATORIO: REQUISITOS DE PREPARACIÓN
Por favor, siga estrictamente las siguientes pautas antes de acudir a la toma de muestras:

1. **Glucosa Basal:**
   * ⚠️ **Requisito Obligatorio:** Requiere un ayuno estricto de 8 a 10 horas. No consuma alimentos ni bebidas (salvo agua pura) durante este lapso.
2. **Hemoglobina Glicosilada (HbA1c):**
   * ✅ **Requisito:** No necesita ayuno. Esta prueba refleja tu promedio de azúcar en la sangre durante los últimos 3 meses.
3. **Examen de Orina y Perfil Lipídico:**
   * ⚠️ **Requisito Obligatorio:** Generalmente requieren de 8 a 12 horas de ayuno previo para no alterar los parámetros de colesterol y triglicéridos.

[Sistema]: Flujo de Urgencias finalizado. Sesión Cerrada.

```

---

### Caso 2: AGENDAR CITA (Glucosa: 110 mg/dL)

* **Datos de Entrada:** Paciente con DNI `12345678` (Juan Pérez).
* **Comportamiento Esperado:** Al estar en rangos estables, se activa el `AppointmentAgent` (Temperatura 0.0). El agente parsea internamente el catálogo `medicos.csv`, ofrece los intervalos de 2 horas según el día elegido por el usuario y finaliza confirmando la cita mediante el tool `confirm_appointment`.

```text
--- BIENVENIDO AL SISTEMA DE TRIAJE DIGITAL (ENDOCRINOLOGÍA) ---

[Sistema]: Paciente localizado: Juan Pérez
[Sistema]: Último registro de glucosa: 110.0 mg/dL

[RESULTADO TRIAJE]: Categoría -> **AGENDAR CITA**
[ACCIÓN DETERMINISTA]: Felicidades: Agendar tu cita de seguimiento

============================================================
 INTERFAZ DE AGENDAMIENTO DE CITAS: Juan Pérez
============================================================

[Agente]: ¡Hola Juan Pérez! Tu nivel de glucosa está en metas de control diario. Vamos a agendar tu cita de seguimiento. Aquí tienes nuestros médicos y horarios. ¿Qué día te convendría asistir (Lunes a Viernes, Sábado o Domingo)?

[Agente]: ¡Perfecto! Aquí tienes la lista de médicos disponibles de Lunes a Viernes:

1. **Dra. Elena Rostova** - Especialidad: Endocrinología Pediátrica
2. **Dr. Alan Grant** - Especialidad: Diabetología Adultos

Los horarios disponibles para estos días son: 07:00, 09:00, 11:00, 13:00, 15:00, 17:00, 19:00.

Por favor, elige un médico y un horario que te convenga.

[Agente]: Excelente elección. Ahora, por favor, indícame el horario que prefieres para tu cita con el Dr. Alan Grant. Los horarios disponibles son: 07:00, 09:00, 11:00, 13:00, 15:00, 17:00, 19:00.

>>> CITA CONFIRMADA: Su espacio ha sido reservado exitosamente. Detalles: Cita con el Dr. Alan Grant el Lunes a las 15:00 <<< 

```

---

### Caso 3: EMERGENCIA - Híbrido Agéntico (Glucosa: 45 mg/dL)

* **Datos de Entrada:** Paciente con DNI `45678912` (Carlos Mendoza).
* **Comportamiento Esperado:** Al detectar una hipoglucemia crítica severa (< 70 mg/dL), se imprime el aviso de emergencia de forma determinista y se despliega el `EmergencyAgent`. El agente sigue el estricto protocolo de tres preguntas clínicas y, tras sugerir mitigación inmediata, ejecuta forzosamente la herramienta `@tool notify_doctor_emergency`.

```text
--- BIENVENIDO AL SISTEMA DE TRIAJE DIGITAL (ENDOCRINOLOGÍA) ---

[Sistema]: Paciente localizado: Carlos Mendoza
[Sistema]: Último registro de glucosa: 45.0 mg/dL

[RESULTADO TRIAJE]: Categoría -> **EMERGENCIA**
[ACCIÓN DETERMINISTA]: Llamar a Emergencias

============================================================
 ALERTA CRÍTICA: INICIANDO CHAT DE EMERGENCIA - Carlos Mendoza
============================================================

[Agente]: Hola Carlos Mendoza, detectamos un nivel de glucosa crítico (45.0 mg/dL). Por favor, indícame qué síntomas tienes ahora mismo.

[Agente]: Gracias por compartir eso, Carlos. Ahora, ¿puedes decirme cuántas veces a la semana has hecho ejercicio recientemente?

[Agente]: Entiendo. Ahora, ¿has consumido carbohidratos en exceso o bebido alcohol recientemente?

[Agente]: Gracias por la información, Carlos. Es importante que tomes medidas inmediatas para elevar tu nivel de glucosa. Por favor, consume 15 gramos de carbohidratos de acción rápida, como una bebida azucarada o tabletas de glucosa.

Voy a notificar a tu médico de cabecera sobre tu estado para que pueda brindarte la atención necesaria.

>>> EMERGENCIA: Los datos proporcionados están siendo notificados a tu médico, se recomienda llamar a emergencias. <<< 

```